In [283]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import re
import math
import os
import glob
from statistics import mean, stdev
import re
from astropy import units as u
from astropy.coordinates import SkyCoord    
import mwdust
import ast
import pickle

In [3]:
sys.path.append("/Users/pnr5sh/Documents/phd/mmmp/")
import sidchaini.sidhelpers as sidhelpers

In [4]:
# read in gopreaux LCs
# fit for date of max
# from bs_ZFTBTS, copy original SN: IAUName, ra, dec, redshift, type, A_V
    # add in fake ztf name, fit-for-max peakt, peakmag, peakfilt
    # update bs_ZTSBTS w/ filenames
# copy a random spectra and assign it gopreaux filenames

In [5]:
# reading in file names of the GP LC's
with open("./maven_data/fnames4maven.txt", "r") as f:
    gp_fnames = f.read().splitlines()

lc_dfs = [pd.read_csv(f'./maven_data/lightcurves/{f}') for f in gp_fnames]

def extract_id(filename):
    match = re.search(r"ZTF(\w+?)s\d", filename)
    return match.group(1) if match else None

# creating list of original SN names from gp fnames
og_sn_ids = [f'SN20{extract_id(f)}' for f in gp_fnames]

In [31]:
# date of max fitting for LCs

# craig's function from adap

def plot_fit_for_max(
    sn_name, # was sn_class
    mjd_array,
    mag_array,
    err_array,
    fit_mjds,
    fit_mags,
    fit_errs,
    inds_to_fit,
    ):
    """
    Takes as input arrays for MJD, mag, and err for a filter
    as well as the guess for the MJD of maximum and an array
    to shift the lightcurve over,
    and returns estimates of the peak MJD and mag at peak
    """

    fig, ax = plt.subplots()

    ax.errorbar(mjd_array, mag_array, yerr=err_array, fmt="o", color="gray", ms=1)
    ax.errorbar(
        fit_mjds,
        fit_mags,
        yerr=fit_errs,
        fmt="o",
        color="blue",
        label="Used in Fitting",
        ms=3,
    )
    if len(mjd_array[inds_to_fit]) > 0:
        plt.ylim(min(mag_array[inds_to_fit]) - 0.5, max(mag_array[inds_to_fit]) + 0.5)
    plt.xlabel("MJD")
    plt.ylabel("Apparent Magnitude")
    plt.title(sn_name)
    plt.legend()
    plt.gca().invert_yaxis()

    # plt.show()

def fit_for_max(df, sn_name, filt, shift_array=[-7, -5, -3, 0, 3, 5, 7], plot=False, offset=0, window=10): #[-3, -2, -1, 0, 1, 2, 3]
    """
    Takes as input arrays for MJD, mag, and err for a filter
    as well as the guess for the MJD of maximum and an array
    to shift the lightcurve over,
    and returns estimates of the peak MJD and mag at peak
    """
    mjd_array = df.loc[df['band']==filt, 'time'].to_numpy()
    mag_array = df.loc[df['band']==filt, 'mag'].to_numpy()
    err_array = df.loc[df['band']==filt, 'magerr'].to_numpy()

    if len(mag_array) < 4:  # == 0:
        print('len(mag)<4')
        return None, None

    initial_guess_mjd_max = mjd_array[np.where((mag_array == min(mag_array)))[0]][0] + offset

    fit_inds = np.where((abs(mjd_array - initial_guess_mjd_max) < 40))[0] #was 30
    if len(fit_inds) < 4:
        print('not enough points +/-30 days from initial guess')
        return None, None

    fit_coeffs = np.polyfit(mjd_array[fit_inds], mag_array[fit_inds], 3)
    guess_phases = np.arange(min(mjd_array[fit_inds]), max(mjd_array[fit_inds]), 1)
    p = np.poly1d(fit_coeffs)
    guess_best_fit = p(guess_phases)

    if len(guess_best_fit) == 0:
        print('len(best_guess)=0')
        return None, None

    guess_mjd_max = guess_phases[np.where((guess_best_fit == min(guess_best_fit)))[0]][0]

    # print(f'GUESS MJD MAX = {guess_mjd_max}')
    ### Do this because the array might not be ordered
    inds_to_fit = np.where((mjd_array > guess_mjd_max - window) & (mjd_array < guess_mjd_max + window))
    # print(f'INDS_TO_FIt = {inds_to_fit}')
    
    if len(inds_to_fit[0]) < 4:
        print('Select a wider date range')
        return None, None

    numdata = len(mjd_array[inds_to_fit])
    numiter = max(int(numdata * np.log(numdata) ** 2), 200)

    fit_mjds = mjd_array[inds_to_fit]
    fit_mags = mag_array[inds_to_fit]
    fit_errs = err_array[inds_to_fit]

    if plot:
        plot_fit_for_max(
            sn_name=sn_name,
            mjd_array=mjd_array,
            mag_array=mag_array,
            err_array=err_array,
            fit_mjds=fit_mjds,
            fit_mags=fit_mags,
            fit_errs=fit_errs,
            inds_to_fit=inds_to_fit,
        )

    peak_mags = []
    peak_mjds = []
    for num in range(numiter):
        simulated_points = []

        ### Shift by a certain number of days to randomly sample the light curve
        sim_shift = np.random.choice(shift_array)

        inds_to_fit = np.where((mjd_array > guess_mjd_max - (window/2) + sim_shift) & (mjd_array < guess_mjd_max + (window/2) + sim_shift))[0]
        if len(inds_to_fit) > 0:

            fit_mjds = mjd_array[inds_to_fit]
            fit_mags = mag_array[inds_to_fit]
            fit_errs = err_array[inds_to_fit]

            for i in range(len(fit_mjds)):
                simulated_points.append(np.random.normal(fit_mags[i], fit_errs[i]))

            fit = np.polyfit(fit_mjds, simulated_points, 2)
            f = np.poly1d(fit)
            fit_time = np.linspace(min(fit_mjds), max(fit_mjds), 100)

            if num % 25 == 0 and plot:
                plt.plot(fit_time, f(fit_time), color="black", linewidth=0.5)
            peak_mag = min(f(fit_time))
            peak_mags.append(peak_mag)
            peak_mjds.append(fit_time[np.argmin(f(fit_time))])

    if len(peak_mjds) == 0:
        print('len(peak_mjds)=0')
        return None, None

    if plot:
        plt.errorbar(
            mean(peak_mjds),
            mean(peak_mags),
            xerr=stdev(peak_mjds),
            yerr=stdev(peak_mags),
            color="red",
            fmt="o",
            label="Best Fit Peak",
        )
        plt.xlim(guess_mjd_max - 30, guess_mjd_max + 30)
        plt.legend()
        plt.show()

    return mean(peak_mjds), mean(peak_mags)

In [32]:
# %matplotlib qt

def fit_max(
        dfs,
        sn_names,
        savefile,
        plot=True,
        first_fit=False,
        refit_r=False,
        refit_g=False,
        inds_to_fit=[],
        save2file=False,
        window=10,
        ):
          
    if first_fit:
        #initialize savefile
        with open(savefile, 'a') as file:
            file.write('#sn_name peak_mjd peak_mag filt\n')
        file.close()

        filt = 'g'

        for i,df in enumerate(dfs):
            sn_name = sn_names[i]
            print(i, sn_name)

            peak_mjd, peak_mag = fit_for_max(df, sn_name, filt,
                                             shift_array=[-3, -2, -1, 0, 1, 2, 3],
                                             plot=plot, offset=0, window=window)
            print(peak_mjd, peak_mag)

            if save2file:
                line = f'{sn_name} {peak_mjd} {peak_mag} {filt}\n'
                with open(savefile, 'a') as file:
                    file.write(line)
                file.close()

    elif refit_r:
        filt = 'R'
        if not inds_to_fit:
            print('Must include indeces to slice on')
            return
        
        for i,df in enumerate(dfs):
            if any(i == x for x in inds_to_fit):
                sn_name = sn_names[i]
                print(i, sn_name)

                peak_mjd, peak_mag = fit_for_max(df, sn_name, filt,
                                                 shift_array=[-3, -2, -1, 0, 1, 2, 3],
                                                 plot=plot, offset=0, window=window)
                print(peak_mjd, peak_mag)

                if save2file:
                    line = f'{sn_name} {peak_mjd} {peak_mag} {filt}\n'
                    with open(savefile, 'a') as file:
                        file.write(line)
                    file.close()

    elif refit_g:
        filt = 'g'
        if not inds_to_fit:
            print('Must include indeces to slice on')
            return
        
        for i,df in enumerate(dfs):
            if any(i == x for x in inds_to_fit):
                sn_name = sn_names[i]
                print(i, sn_name)

                peak_mjd, peak_mag = fit_for_max(df, sn_name, filt,
                                                 shift_array=[-3, -2, -1, 0, 1, 2, 3],
                                                 plot=plot, offset=0, window=window)
                print(peak_mjd, peak_mag)

                if save2file:
                    line = f'{sn_name} {peak_mjd} {peak_mag} {filt}\n'
                    with open(savefile, 'a') as file:
                        file.write(line)
                    file.close()

In [ ]:
# Don't need to re-run #########################################

# #fitting max of LC for Ib

# savefile = './temp/peak_fit_gp_objs.txt'

# refit_g_inds = [0,1,2,3,4,5,6,7,9,10,11,12,13,]
# # refit_r_inds = []

# # %matplotlib qt
# fit_max(lc_dfs, gp_fnames, savefile, plot=True, first_fit=False,
#         refit_r=True, refit_g=False, inds_to_fit=refit_g_inds,
#         save2file=False, window=80) #window = 15 for first fit

0 ZTF20adnxs21_0.csv
59486.930004513815 17.861349429495167
1 ZTF20ikqs10x_0.csv
59302.024451058256 19.875419526622863
2 ZTF20ikqs12x_0.csv
58573.69646587581 18.83593688923109
3 ZTF20ikqs13x_0.csv
58066.99556400723 18.95144429988635
4 ZTF20ikqs02x_0.csv
58062.320524530434 19.32608690154801
5 ZTF20ikqs06x_0.csv
59509.795688154445 17.952637341877058
6 ZTF20rscs12x_0.csv
59554.6050368617 21.31666459374482
7 ZTF20rscs15x_0.csv
59273.17149271479 21.5613025256501
9 ZTF20rscs21x_0.csv
59205.4141106155 20.371221896866338
10 ZTF20rscs22x_0.csv
59234.806765436675 21.497362962551996
11 ZTF20rscs02x_0.csv
59698.61538008694 19.51991236358881
12 ZTF20rscs05x_0.csv
58226.57435438705 21.073679030830398
13 ZTF20rscs07x_0.csv
60061.94924342435 22.235779536948538


In [116]:
# reading in the bs_ZTFBTS table that has only real LC objects
bs_ztf_df = pd.read_csv('./maven_data/bs_ZTFBTS_TransientTable.csv')

In [ ]:
# checking which of the GP objects are not based on objs already in the maven dataset
    # will need to find ra, dec, redshift, for these objects 
    # will need to append type IIb and calculate A_V as well
need_info = {x: x not in bs_ztf_df['IAUID'].values for x in og_sn_ids}

# for item in need_info:
#     if need_info[item]:
#         print(item)

#SN, RA, Dec, Z --- from peaky finders paper

# SN2020ikq, 13.6013933, 28.9833639, 0.037
# SN2020rsc, 1.3323619, 38.1860167, 0.0313
# SN2020sbw, 2.7675883, 3.3299056, 0.023033
# SN2021pb, 9.7463333, 51.6873889, 0.033
# SN2022qzr, 0.1652781, -5.0211361, 0.018705


need_info['SN2020ikq'] = {'ra': 13.6013933, 'dec': 28.9833639, 'z': 0.037}
need_info['SN2020rsc'] = {'ra': 1.3323619, 'dec': 38.1860167, 'z': 0.0313}
need_info['SN2020sbw'] = {'ra': 2.7675883, 'dec': 3.3299056, 'z':0.023033}
need_info['SN2021pb'] = {'ra': 9.7463333, 'dec': 51.6873889, 'z':0.033}
need_info['SN2022qzr'] = {'ra': 0.1652781, 'dec': -5.0211361, 'z':0.018705}

SN2020ikq
SN2020rsc
SN2020sbw
SN2021pb
SN2022qzr


In [ ]:
#calculating extinction
r_v = 3.1

for sn, data in need_info.items():
    if data:
        sfd = mwdust.SFD()
        coord = SkyCoord(data['ra'] * u.deg, data['dec'] * u.deg).galactic
        av = r_v*sfd(coord.l.value, coord.b.value, 5000)[0]
        data['a_v'] = float(av)

In [117]:
fit4peak_results = pd.read_csv('./temp/peak_fit_gp_objs.txt', skiprows=1, names=['sn_name', 'peak_mjd', 'peak_mag', 'filt'], delimiter=' ')

for i,ztfsn in enumerate(gp_fnames):
    iausn = og_sn_ids[i]
    # col_names are ['ZTFID', 'IAUID', 'RA', 'Dec', 'peakt', 'peakfilt', 'peakmag', 'type', 'redshift', 'double-peaked', 'A_V', 'filenames']
    if not need_info[iausn]:
        # print(iausn, 'already has data in bs ztf')
        bs_ztf_df.loc[len(bs_ztf_df)] = [ztfsn[:-6], 
                                         iausn, 
                                         bs_ztf_df.loc[bs_ztf_df['IAUID']==iausn, 'RA'].iloc[0],
                                         bs_ztf_df.loc[bs_ztf_df['IAUID']==iausn, 'Dec'].iloc[0],
                                         fit4peak_results.loc[fit4peak_results['sn_name']==ztfsn, 'peak_mjd'].iloc[-1],
                                         fit4peak_results.loc[fit4peak_results['sn_name']==ztfsn, 'filt'].iloc[-1],
                                         fit4peak_results.loc[fit4peak_results['sn_name']==ztfsn, 'peak_mag'].iloc[-1],
                                         'SN IIb',
                                         bs_ztf_df.loc[bs_ztf_df['IAUID']==iausn, 'redshift'].iloc[0],
                                         bs_ztf_df.loc[bs_ztf_df['IAUID']==iausn, 'double-peaked'].iloc[0],
                                         bs_ztf_df.loc[bs_ztf_df['IAUID']==iausn, 'A_V'].iloc[0],
                                         [ztfsn[:-4]],
                                        ]
    else:
        # print(iausn, 'read in from need_info dict')
        bs_ztf_df.loc[len(bs_ztf_df)] = [ztfsn[:-6], 
                                         iausn, 
                                         need_info[iausn]['ra'],
                                         need_info[iausn]['dec'],
                                         fit4peak_results.loc[fit4peak_results['sn_name']==ztfsn, 'peak_mjd'].iloc[-1],
                                         fit4peak_results.loc[fit4peak_results['sn_name']==ztfsn, 'filt'].iloc[-1],
                                         fit4peak_results.loc[fit4peak_results['sn_name']==ztfsn, 'peak_mag'].iloc[-1],
                                         'SN IIb',
                                         need_info[iausn]['z'],
                                         1.0,
                                         need_info[iausn]['a_v'],
                                         [ztfsn[:-4]],
                                        ]

# matching case of pre-existing ZTFBTS file        
bs_ztf_df['peakfilt'] = bs_ztf_df['peakfilt'].str.lower()

In [289]:
# saving to new ZFTBTS file to give to maven
bs_ztf_df_sorted = bs_ztf_df.sort_values(by='ZTFID', ignore_index=True)
# bs_ztf_df.to_csv('./maven_data/bs_ZTFBTS_TransientTable_w_gp_objs.csv', index=False)
bs_ztf_df_sorted.to_csv('./maven_data/bs_ZTFBTS_TransientTable_w_gp_objs_sorted.csv', index=False)

In [164]:
# creating copies to a spectra to rename based on new GP objs 
    # note: this is fine for now b/c we are assuming spectra hold no prediction power w maven based on initial fine-tuning runs
    # going with ZTF23aaialcw_0 i.e. SN 2023gwl as the one to copy

spec2copy = pd.read_csv('./maven_data/spectra/ZTF23aaialcw_0.csv', names=['wavelength', 'flux'])
for filename in gp_fnames:
    spec2copy.to_csv(f'./maven_data/spectra/{filename}', index=False, header=False)

In [124]:
#class balance b/w single and double peaked
print(len(bs_ztf_df.loc[bs_ztf_df['double-peaked']==0])/len(bs_ztf_df), len(bs_ztf_df.loc[bs_ztf_df['double-peaked']==1])/len(bs_ztf_df))
print(len(bs_ztf_df.loc[bs_ztf_df['double-peaked']==0]), len(bs_ztf_df.loc[bs_ztf_df['double-peaked']==1]), len(bs_ztf_df))

0.6206896551724138 0.3793103448275862
90 55 145


In [ ]:
#################################
#
#
#     creating k-fold splits
#
#
#################################

In [189]:
#NOTE: maven orders filenames alphanumerically, 
    # so we need to update bs_ZTFBTS / filenames to get correct indeces for kfold

bs_ztf_df_sorted = bs_ztf_df.sort_values(by='ZTFID', ignore_index=True)

In [180]:
bs_ztf_df_sorted.to_csv('./temp/bs_ZTFBTS_TransientTable_sorted_w_gp.csv', index=False)

In [ ]:
kfold1 = bs_ztf_df_sorted.iloc[0:13][['double-peaked', 'filenames']]  
kfold1a = bs_ztf_df_sorted.iloc[129:][['double-peaked', 'filenames']]
kfold1_sum = pd.concat([kfold1, kfold1a],ignore_index=True)                             # 8 dp

kfold2 = bs_ztf_df_sorted.iloc[13:23][['double-peaked', 'filenames']] # 1 dp
kfold2a = bs_ztf_df_sorted.iloc[112:129][['double-peaked', 'filenames']] # 5 dp
kfold2_sum = pd.concat([kfold2, kfold2a])                                               # 6 dp

kfold3 = bs_ztf_df_sorted.iloc[23:35][['double-peaked', 'filenames']] # 3 dp
kfold3a = bs_ztf_df_sorted.iloc[96:112][['double-peaked', 'filenames']] # 11 dp
kfold3_sum = pd.concat([kfold3, kfold3a])                                               # 14 dp

kfold4_sum = bs_ztf_df_sorted.iloc[35:67][['double-peaked', 'filenames']]               # 16 dp

kfold5_sum = bs_ztf_df_sorted.iloc[67:96][['double-peaked', 'filenames']]               # 14 dp

In [ ]:
#print len of filenames per kfold
len([item for entry in kfold5_sum.filenames.to_list() for item in (ast.literal_eval(entry) if isinstance(entry, str) else entry)])

30

In [278]:
# getting the correct indeces for the train / test split dict

# create zipped list of i,ztf_filename
# then iterate thru the kfold filenames, matching in filename, and appending the i from the zipped list to index list

all_file_names = [item for entry in bs_ztf_df_sorted.filenames.to_list() for item in (ast.literal_eval(entry) if isinstance(entry, str) else entry)]
fnames_df = pd.DataFrame(list(zip(range(len(all_file_names)),all_file_names)), columns=['index', 'fname'])

def get_kfold_indeces(kfold_df):
    flat_names_kfold = [item for entry in kfold_df.filenames.to_list() for item in (ast.literal_eval(entry) if isinstance(entry, str) else entry)]
    indeces = fnames_df.loc[fnames_df['fname'].isin(flat_names_kfold),'index'].to_list()
    return indeces

In [282]:
kfold1_idx = get_kfold_indeces(kfold1_sum)
kfold2_idx = get_kfold_indeces(kfold2_sum)
kfold3_idx = get_kfold_indeces(kfold3_sum)
kfold4_idx = get_kfold_indeces(kfold4_sum)
kfold5_idx = get_kfold_indeces(kfold5_sum)

all_idx = range(len(fnames_df))

In [284]:
#saving the train, test indices as tuples in array to load into maven
kfold_split_dict = {
    'kfold1': ([x for x in all_idx if x not in kfold1_idx], kfold1_idx),
    'kfold2': ([x for x in all_idx if x not in kfold2_idx], kfold2_idx),
    'kfold3': ([x for x in all_idx if x not in kfold3_idx], kfold3_idx),
    'kfold4': ([x for x in all_idx if x not in kfold4_idx], kfold4_idx),
    'kfold5': ([x for x in all_idx if x not in kfold5_idx], kfold5_idx),
}

with open('maven_data/kfold_split_dict_gp_objs.pkl', 'wb+') as f:
    pickle.dump(kfold_split_dict, f)

In [298]:
# kfold1 = bs_ztf_df.iloc[0:21][['double-peaked', 'filenames']]
# kfold2 = bs_ztf_df.iloc[21:37][['double-peaked', 'filenames']]
# kfold3 = bs_ztf_df.iloc[37:56][['double-peaked', 'filenames']]
# kfold4 = bs_ztf_df.iloc[56:76][['double-peaked', 'filenames']]
# kfold5 = bs_ztf_df.iloc[76:99][['double-peaked', 'filenames']]

# kfold1a = bs_ztf_df.iloc[113:123][['double-peaked', 'filenames']]
# kfold2a = bs_ztf_df.iloc[123:133][['double-peaked', 'filenames']]
# kfold3a = bs_ztf_df.iloc[105:113][['double-peaked', 'filenames']]
# kfold3b = bs_ztf_df.iloc[133:135][['double-peaked', 'filenames']]
# kfold4a = bs_ztf_df.iloc[99:100][['double-peaked', 'filenames']] 
# kfold4b = bs_ztf_df.iloc[138:144][['double-peaked', 'filenames']] 
# kfold5a = bs_ztf_df.iloc[100:105][['double-peaked', 'filenames']]
# kfold5b = bs_ztf_df.iloc[135:138][['double-peaked', 'filenames']]
# kfold5c = bs_ztf_df.iloc[144:145][['double-peaked', 'filenames']]


# kfold1_sum = pd.concat([kfold1, kfold1a])
# kfold2_sum = pd.concat([kfold2, kfold2a])
# kfold3_sum = pd.concat([kfold3,kfold3a,kfold3b])
# kfold4_sum = pd.concat([kfold4, kfold4a,kfold4b])
# kfold5_sum = pd.concat([kfold5, kfold5a, kfold5b, kfold5c])

# kfold5

In [ ]:
################################################################
#
#
#       50/50 split + 3 kfolds
#
#
################################################################


In [302]:
to_kill_sp = ['SN2018fzn', 'SN2018jee', 'SN2018iqx', 'SN2018iug', 'SN2018jaw', 'SN2019dwa', 'SN2019buy', 'SN2019abb', 'SN2018kva',
              'SN2019dxr', 'SN2019gaf', 'SN2019uff', 'SN2019pof', 'SN2019lfj', 'SN2019yxp', 'SN2020abdw', 'SN2020aut', 'SN2020bmj',
              'SN2020kxg', 'SN2021bm', 'SN2021krf', 'SN2021kww', 'SN2021vis', 'SN2021uth', 'SN2021vnw', 'SN2022abuj', 'SN2022lfa',
              'SN2022nmq', 'SN2023ijh', 'SN2022wwd', 'SN2022rbo', 'SN2022qdg', 'SN2023lya', 'SN2023usf', 'SN2023wf', 'SN2024ambk',
              'SN2024afbf', 'SN2024afao', 'SN2024abtu']

In [ ]:
full_dataset = pd.read_csv('./maven_data/bs_ZTFBTS_TransientTable_w_gp_objs_sorted.csv')
full_dataset

,ZTFID,IAUID,RA,Dec,peakt,peakfilt,peakmag,type,redshift,double-peaked,A_V,filenames
0,ZTF18aahhzqn,SN2018avy,178.754792,32.075350,58225.156007,r,18.147050,SN Ib/c,0.031000,0.0,0.069504,['ZTF18aahhzqn_0']
1,ZTF18aahuujv,SN2019ilo,200.476341,32.422029,58673.730376,g,18.691914,SN Ic,0.034300,0.0,0.061298,['ZTF18aahuujv_0']
2,ZTF18aakuewf,SN2018bcc,243.594375,35.917889,58231.129356,g,17.692159,SN Ibn,0.063600,0.0,0.043482,['ZTF18aakuewf_0']
3,ZTF18abojpnr,SN2018fzn,297.487120,59.592827,58370.051276,g,19.171934,SN IIb,0.037500,0.0,0.258507,"['ZTF18abojpnr_0', 'ZTF18abojpnr_1']"
4,ZTF18absliyc,SN2018gcj,6.005904,-1.223377,58371.685414,g,18.892202,SN Ic,0.080000,0.0,0.086336,['ZTF18absliyc_0']
...,...,...,...,...,...,...,...,...,...,...,...,...
140,ZTF24abtnkbi,SN2024abtu,108.451121,50.257581,60649.346250,g,17.680278,SN Ic,0.017682,0.0,0.211343,['ZTF24abtnkbi_0']
141,ZTF24abzgggz,SN2024afbf,170.447324,46.128450,60675.269521,g,19.457773,SN Ic,0.030000,0.0,0.054166,['ZTF24abzgggz_0']
142,ZTF24abzmhtv,SN2024afao,44.505938,41.713999,60683.357876,g,18.691380,SN Ic,0.050000,0.0,0.330174,['ZTF24abzmhtv_0']
143,ZTF24zsws21x,SN2024zsw,20.793617,19.570172,60677.119596,g,19.585561,SN IIb,0.032286,1.0,0.161435,['ZTF24zsws21x_0']


In [316]:
smoler_dataset = full_dataset.loc[~(full_dataset['IAUID'].isin(to_kill_sp))].reset_index()
smoler_dataset.to_csv('./temp/get_rid_of_sp_dataset.csv', index=False)

#drop some of the extra spectra for two of the sp objects
smoler_dataset.loc[smoler_dataset['IAUID']=='SN2019hgp', 'filenames'] = "['ZTF19aayejww_0', 'ZTF19aayejww_1']"
smoler_dataset.loc[smoler_dataset['IAUID']=='SN2022oqm', 'filenames'] = "['ZTF22aasxgjp_0', 'ZTF22aasxgjp_1', 'ZTF22aasxgjp_2']"

# checking numbers 
smoler_dataset_sp = smoler_dataset.loc[smoler_dataset['double-peaked']==0]
smoler_dataset_dp = smoler_dataset.loc[smoler_dataset['double-peaked']==1]
print('sp:',len([item for entry in smoler_dataset_sp.filenames.to_list() for item in (ast.literal_eval(entry) if isinstance(entry, str) else entry)]),
      'dp:',len([item for entry in smoler_dataset_dp.filenames.to_list() for item in (ast.literal_eval(entry) if isinstance(entry, str) else entry)]))

# save this reduced dataset to maven_data
smoler_dataset = smoler_dataset.drop(columns='index')
smoler_dataset.to_csv('./maven_data/bs_ZTFBTS_TransientTable_w_gp_objs_sorted_equal_split.csv', index=False)

sp: 58 dp: 58


In [317]:
##################################
#
#     creating 3-way kfolds
#
###################################

In [342]:
# we have 116 objects, 58 of each kind
    # 39, 39, 38 per fold
    # roughly 19 of each type per kfold

kfold1 = smoler_dataset.iloc[0:18][['double-peaked', 'filenames']]                  # 19 sp, 1 dp  
kfold1a = smoler_dataset.iloc[40:58][['double-peaked', 'filenames']]                # 0 sp, 18 dp
kfold1_sum = pd.concat([kfold1, kfold1a])                                           # 19 + 19 = 38 objs

kfold2 = smoler_dataset.iloc[18:31][['double-peaked', 'filenames']]               # 14 sp, 3 dp
kfold2a = smoler_dataset.iloc[32:40][['double-peaked', 'filenames']]              # 1 sp, 7 dp
kfold2b = smoler_dataset.iloc[94:][['double-peaked', 'filenames']]                # 5 sp, 8 dp
kfold2c = smoler_dataset.iloc[89:90][['double-peaked', 'filenames']]              # 1 dp (2023gwl)
kfold2_sum = pd.concat([kfold2, kfold2a, kfold2b, kfold2c])                       # 20 + 19 = 39 objs

kfold3 = smoler_dataset.iloc[58:89][['double-peaked', 'filenames']]               # 16 sp, 18 dp
kfold3a = smoler_dataset.iloc[90:94][['double-peaked', 'filenames']]              # 3 sp, 1 dp
kfold3b = smoler_dataset.iloc[31:32][['double-peaked', 'filenames']]              # 1 dp (2020urc)
kfold3_sum = pd.concat([kfold3, kfold3a, kfold3b])                                # 19 + 20 = 39 objs

In [ ]:
# getting the correct indeces for the train / test split dict

# create zipped list of i,ztf_filename
# then iterate thru the kfold filenames, matching in filename, and appending the i from the zipped list to index list

all_file_names = [item for entry in smoler_dataset.filenames.to_list() for item in (ast.literal_eval(entry) if isinstance(entry, str) else entry)]
fnames_df = pd.DataFrame(list(zip(range(len(all_file_names)),all_file_names)), columns=['index', 'fname'])

kfold1_idx = get_kfold_indeces(kfold1_sum)
kfold2_idx = get_kfold_indeces(kfold2_sum)
kfold3_idx = get_kfold_indeces(kfold3_sum)

all_idx = range(len(fnames_df))

In [407]:
# fnames_df['fname'].to_list()

In [349]:
#saving the train, test indices as tuples in array to load into maven
kfold_split_dict = {
    'kfold1': ([x for x in all_idx if x not in kfold1_idx], kfold1_idx),
    'kfold2': ([x for x in all_idx if x not in kfold2_idx], kfold2_idx),
    'kfold3': ([x for x in all_idx if x not in kfold3_idx], kfold3_idx),
}

with open('maven_data/kfold_split_dict_gp_objs_equal_split.pkl', 'wb+') as f:
    pickle.dump(kfold_split_dict, f)

In [352]:
##################################################
#
#     creating 3-way kfolds + random sp dropped
#
###################################################

In [375]:
full_dataset = pd.read_csv('./temp/bs_ZTFBTS_TransientTable_sorted_w_gp--allrows.csv')
full_dataset

,ZTFID,IAUID,RA,Dec,peakt,peakfilt,peakmag,type,redshift,double-peaked,A_V,filenames
0,ZTF18aahhzqn,SN2018avy,178.754792,32.075350,58225.156007,r,18.147050,SN Ib/c,0.031000,0.0,0.069504,ZTF18aahhzqn_0
1,ZTF18aahuujv,SN2019ilo,200.476341,32.422029,58673.730376,g,18.691914,SN Ic,0.034300,0.0,0.061298,ZTF18aahuujv_0
2,ZTF18aakuewf,SN2018bcc,243.594375,35.917889,58231.129356,g,17.692159,SN Ibn,0.063600,0.0,0.043482,ZTF18aakuewf_0
3,ZTF18abojpnr,SN2018fzn,297.487120,59.592827,58370.051276,g,19.171934,SN IIb,0.037500,0.0,0.258507,ZTF18abojpnr_0
4,ZTF18abojpnr,SN2018fzn,297.487120,59.592827,58370.051276,g,19.171934,SN IIb,0.037500,0.0,0.258507,ZTF18abojpnr_1
...,...,...,...,...,...,...,...,...,...,...,...,...
160,ZTF24abtnkbi,SN2024abtu,108.451121,50.257581,60649.346250,g,17.680278,SN Ic,0.017682,0.0,0.211343,ZTF24abtnkbi_0
161,ZTF24abzgggz,SN2024afbf,170.447324,46.128450,60675.269521,g,19.457773,SN Ic,0.030000,0.0,0.054166,ZTF24abzgggz_0
162,ZTF24abzmhtv,SN2024afao,44.505938,41.713999,60683.357876,g,18.691380,SN Ic,0.050000,0.0,0.330174,ZTF24abzmhtv_0
163,ZTF24zsws21x,SN2024zsw,20.793617,19.570172,60677.119596,g,19.585561,SN IIb,0.032286,1.0,0.161435,ZTF24zsws21x_0


In [383]:
sp_objs = full_dataset.loc[full_dataset['double-peaked']==0]

# Shuffle a copy to ensure random (not phase-ordered) selection
rng = np.random.default_rng(0)
shuffled_sp_df = sp_objs.sample(frac=1, random_state=rng.integers(1e9)).reset_index(drop=True)

random_kill = shuffled_sp_df['filenames'].sample(n=49)
random_kill #removed by hand

random_drop_df = pd.read_csv('temp/bs_ZTFBTS_TransientTable_sorted_w_gp--random_drop.csv')

# checking numbers 
random_drop_df_sp = random_drop_df.loc[random_drop_df['double-peaked']==0]
random_drop_df_dp = random_drop_df.loc[random_drop_df['double-peaked']==1]
print('sp:',len([item for entry in random_drop_df_sp.filenames.to_list() for item in (ast.literal_eval(entry) if isinstance(entry, str) else entry)]),
      'dp:',len([item for entry in random_drop_df_dp.filenames.to_list() for item in (ast.literal_eval(entry) if isinstance(entry, str) else entry)]))

sp: 58 dp: 58


In [408]:
# we have 116 objects, 58 of each kind
    # 39, 39, 38 per fold
    # roughly 19 of each type per kfold

kfold1 = random_drop_df.iloc[0:18][['double-peaked', 'filenames']]                  # 19 sp, 1 dp  
kfold1a = random_drop_df.iloc[41:59][['double-peaked', 'filenames']]                # 0 sp, 18 dp
kfold1_sum = pd.concat([kfold1, kfold1a])                                           # 19 + 19 = 38 objs

kfold2 = random_drop_df.iloc[18:32][['double-peaked', 'filenames']]               # 14 sp, 3 dp
kfold2a = random_drop_df.iloc[33:41][['double-peaked', 'filenames']]              # 1 sp, 7 dp
kfold2b = random_drop_df.iloc[92:][['double-peaked', 'filenames']]                # 4 sp, 8 dp
kfold2c = random_drop_df.iloc[88:89][['double-peaked', 'filenames']]              # 0 sp, 1 dp (2023gwl)
kfold2_sum = pd.concat([kfold2, kfold2a, kfold2b, kfold2c])                       # 19 + 19 = 38 objs

kfold3 = random_drop_df.iloc[59:88][['double-peaked', 'filenames']]               # 18 sp, 18 dp
kfold3a = random_drop_df.iloc[32:33][['double-peaked', 'filenames']]              # 0 sp, 1 dp (2020urc)
kfold3b = random_drop_df.iloc[89:92][['double-peaked', 'filenames']]              # 2 sp, 1 dp
kfold3_sum = pd.concat([kfold3, kfold3a])                                         # 20 + 20 = 40 objs

#watch out for 2020urc, and 2023gwl!!!

In [409]:
# getting the correct indeces for the train / test split dict

# create zipped list of i,ztf_filename
# then iterate thru the kfold filenames, matching in filename, and appending the i from the zipped list to index list

all_file_names = [item for entry in random_drop_df.filenames.to_list() for item in (ast.literal_eval(entry) if isinstance(entry, str) else entry)]
fnames_df = pd.DataFrame(list(zip(range(len(all_file_names)),all_file_names)), columns=['index', 'fname'])

kfold1_idx = get_kfold_indeces(kfold1_sum)
kfold2_idx = get_kfold_indeces(kfold2_sum)
kfold3_idx = get_kfold_indeces(kfold3_sum)

all_idx = range(len(fnames_df))

In [ ]:
#saving the train, test indices as tuples in array to load into maven
kfold_split_dict = {
    'kfold1': ([x for x in all_idx if x not in kfold1_idx], kfold1_idx),
    'kfold2': ([x for x in all_idx if x not in kfold2_idx], kfold2_idx),
    'kfold3': ([x for x in all_idx if x not in kfold3_idx], kfold3_idx),
}

with open('maven_data/kfold_split_dict_gp_objs_equal_split_random_drop.pkl', 'wb+') as f:
    pickle.dump(kfold_split_dict, f)

# hand-saved the ZTF csv info file to maven_data

In [411]:
fnames_df['fname'].to_list()

['ZTF18aahhzqn_0',
 'ZTF18aahuujv_0',
 'ZTF18absliyc_0',
 'ZTF18acisruf_0',
 'ZTF18acnmifq_0',
 'ZTF18acrcyqw_0',
 'ZTF18aczqzrj_0',
 'ZTF19aadwtoe_0',
 'ZTF19aadwtoe_1',
 'ZTF19aalouag_0',
 'ZTF19aamsetj_0',
 'ZTF19aanijpu_0',
 'ZTF19aarfyvc_0',
 'ZTF19aarnqys_0',
 'ZTF19aayejww_0',
 'ZTF19aayejww_2',
 'ZTF19abafmwj_0',
 'ZTF19abqmsnk_0',
 'ZTF19abvdgqo_0',
 'ZTF19aceshib_0',
 'ZTF19ackjjwf_0',
 'ZTF19ackjszs_0',
 'ZTF19ackjszs_1',
 'ZTF19ackjszs_2',
 'ZTF19acmelor_0',
 'ZTF20aaekkuv_0',
 'ZTF20aahggbm_0',
 'ZTF20aahggbm_2',
 'ZTF20aakodyh_0',
 'ZTF20aalcyih_0',
 'ZTF20aammtwx_0',
 'ZTF20aaurfnl_0',
 'ZTF20abbhrrt_0',
 'ZTF20abeezca_0',
 'ZTF20abkiarz_0',
 'ZTF20abxpoxd_0',
 'ZTF20acfqngt_0',
 'ZTF20acgiglu_0',
 'ZTF20actpqgc_0',
 'ZTF20adadlqm_0',
 'ZTF20adnxs21_0',
 'ZTF20ikqs02x_0',
 'ZTF20ikqs06x_0',
 'ZTF20ikqs10x_0',
 'ZTF20ikqs12x_0',
 'ZTF20ikqs13x_0',
 'ZTF20rscs02x_0',
 'ZTF20rscs05x_0',
 'ZTF20rscs07x_0',
 'ZTF20rscs12x_0',
 'ZTF20rscs15x_0',
 'ZTF20rscs18x_0',
 'ZTF20rscs2